<a href="https://colab.research.google.com/github/julianocmachado/uci-motion-rnn/blob/main/notebooks/01_exploracao/03_carregamento_dataset_completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 1 — Notebook 3: carregamento e organização do dataset completo

Nos notebooks anteriores, carregamos e exploramos os sinais do **usuário 1**. Agora, vamos generalizar esse procedimento para todos os indivíduos e experimentos da base *Smartphone-Based Recognition of Human Activities and Postural Transitions*.

Ao final, teremos os sinais, os rótulos e os metadados necessários para preparar as entradas de uma RNN nos próximos notebooks. Nesta etapa, ainda **não** faremos normalização, janelamento ou divisão entre treino e teste.

## 1. Montagem do Google Drive

A base deve estar na mesma pasta usada nos Notebooks 1 e 2.

In [19]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Importação das bibliotecas

- `Path`: construção dos caminhos de pastas e arquivos;
- `re`: extração dos números do experimento e do usuário a partir dos nomes dos arquivos;
- `NumPy`: carregamento e organização numérica dos sinais;
- `pandas`: criação das tabelas de rótulos, resumo e distribuição.

In [20]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

## 3. Caminhos e constantes

Os arquivos processados serão gravados em uma subpasta chamada `DadosProcessados`, sem modificar os arquivos originais.

In [21]:
PASTA_BASE = Path('/content/drive/MyDrive/UCI-Motion-Database')
PASTA_RAW = PASTA_BASE / 'RawData'
PASTA_SAIDA = PASTA_BASE / 'DadosProcessados'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

FREQUENCIA_AMOSTRAGEM = 50  # Hz

NOMES_CARACTERISTICAS = [
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
]

NOMES_ATIVIDADES = {
    0: 'sem rótulo',
    1: 'ficar em pé',
    2: 'sentar',
    3: 'deitar',
    4: 'caminhar',
    5: 'subir escadas',
    6: 'descer escadas',
    7: 'sentar → ficar em pé',
    8: 'ficar em pé → sentar',
    9: 'deitar → ficar em pé',
    10: 'ficar em pé → deitar',
    11: 'deitar → sentar',
    12: 'sentar → deitar'
}

if not PASTA_RAW.exists():
    raise FileNotFoundError(f'Pasta não encontrada: {PASTA_RAW}')

print('Pasta de entrada:', PASTA_RAW)
print('Pasta de saída:', PASTA_SAIDA)

Pasta de entrada: /content/drive/MyDrive/UCI-Motion-Database/RawData
Pasta de saída: /content/drive/MyDrive/UCI-Motion-Database/DadosProcessados


## 4. Carregamento da tabela de rótulos

Cada linha de `labels.txt` informa o experimento, o usuário, a atividade e os limites inicial e final de um segmento. Os índices do arquivo começam em 1.

In [22]:
NOMES_COLUNAS_ROTULOS = [
    'experimento', 'usuario', 'atividade', 'inicio', 'fim'
]

rotulos = pd.read_csv(
    PASTA_RAW / 'labels.txt',
    sep=r'\s+',
    names=NOMES_COLUNAS_ROTULOS
)

print('Quantidade de segmentos anotados:', len(rotulos))
print('Quantidade de usuários:', rotulos['usuario'].nunique())
print('Quantidade de experimentos:', rotulos['experimento'].nunique())
rotulos.head()

Quantidade de segmentos anotados: 1214
Quantidade de usuários: 30
Quantidade de experimentos: 61


,experimento,usuario,atividade,inicio,fim
0,1,1,5,250,1232
1,1,1,7,1233,1392
2,1,1,4,1393,2194
3,1,1,8,2195,2359
4,1,1,5,2360,3374


## 5. Identificação automática dos arquivos

Em vez de escrever manualmente os 61 números de experimentos, extrairemos o experimento e o usuário de nomes como `acc_exp01_user01.txt`.

As chaves dos dicionários abaixo são tuplas no formato `(experimento, usuário)`. Assim, conseguimos verificar se cada acelerômetro possui o giroscópio correspondente.

In [23]:
PADRAO_ARQUIVO = re.compile(
    r'^(acc|gyro)_exp(\d+)_user(\d+)\.txt$'
)

arquivos_acc = {}
arquivos_gyro = {}

for caminho in sorted(PASTA_RAW.glob('*.txt')):
    resultado = PADRAO_ARQUIVO.match(caminho.name)

    if resultado is None:
        continue

    sensor, experimento, usuario = resultado.groups()
    chave = (int(experimento), int(usuario))

    if sensor == 'acc':
        arquivos_acc[chave] = caminho
    else:
        arquivos_gyro[chave] = caminho

chaves_acc = set(arquivos_acc)
chaves_gyro = set(arquivos_gyro)
pares_disponiveis = sorted(chaves_acc & chaves_gyro)

print(list(arquivos_acc.values()))
print('Arquivos de acelerômetro:', len(arquivos_acc))
print('Arquivos de giroscópio:', len(arquivos_gyro))
print('Pares disponíveis:', len(pares_disponiveis))

[PosixPath('/content/drive/MyDrive/UCI-Motion-Database/RawData/acc_exp01_user01.txt'), PosixPath('/content/drive/MyDrive/UCI-Motion-Database/RawData/acc_exp02_user01.txt'), PosixPath('/content/drive/MyDrive/UCI-Motion-Database/RawData/acc_exp03_user02.txt'), PosixPath('/content/drive/MyDrive/UCI-Motion-Database/RawData/acc_exp04_user02.txt'), PosixPath('/content/drive/MyDrive/UCI-Motion-Database/RawData/acc_exp05_user03.txt'), PosixPath('/content/drive/MyDrive/UCI-Motion-Database/RawData/acc_exp06_user03.txt'), PosixPath('/content/drive/MyDrive/UCI-Motion-Database/RawData/acc_exp07_user04.txt'), PosixPath('/content/drive/MyDrive/UCI-Motion-Database/RawData/acc_exp08_user04.txt'), PosixPath('/content/drive/MyDrive/UCI-Motion-Database/RawData/acc_exp09_user05.txt'), PosixPath('/content/drive/MyDrive/UCI-Motion-Database/RawData/acc_exp10_user05.txt'), PosixPath('/content/drive/MyDrive/UCI-Motion-Database/RawData/acc_exp11_user06.txt'), PosixPath('/content/drive/MyDrive/UCI-Motion-Database

## 6. Verificação dos pares de arquivos

Se um sensor estiver ausente, o respectivo experimento não poderá ser montado com as seis características.

In [24]:
acc_sem_gyro = sorted(chaves_acc - chaves_gyro)
gyro_sem_acc = sorted(chaves_gyro - chaves_acc)

print('Acelerômetros sem giroscópio:', acc_sem_gyro)
print('Giroscópios sem acelerômetro:', gyro_sem_acc)

assert not acc_sem_gyro, 'Existem arquivos de acelerômetro sem giroscópio.'
assert not gyro_sem_acc, 'Existem arquivos de giroscópio sem acelerômetro.'
assert len(pares_disponiveis) == 61, (
    'Esperavam-se 61 pares de arquivos, mas foram encontrados '
    f'{len(pares_disponiveis)}.'
)

Acelerômetros sem giroscópio: []
Giroscópios sem acelerômetro: []


## 7. Função para carregar e rotular um experimento

A função realiza quatro tarefas:

1. carrega acelerômetro e giroscópio;
2. verifica se os dois sensores possuem o mesmo número de amostras;
3. combina os sensores em uma matriz com seis colunas;
4. cria um rótulo para cada amostra a partir de `labels.txt`.

A classe 0 permanece reservada às amostras que não pertencem a nenhum intervalo anotado. O vetor `cobertura` permite detectar uma eventual sobreposição entre segmentos.

In [25]:
def carregar_experimento_completo(
    caminho_acc,
    caminho_gyro,
    numero_experimento,
    numero_usuario,
    tabela_rotulos
):
    acc = np.loadtxt(caminho_acc, dtype=np.float32)
    gyro = np.loadtxt(caminho_gyro, dtype=np.float32)

    if acc.ndim != 2 or acc.shape[1] != 3:
        raise ValueError(f'Formato inválido em {caminho_acc.name}: {acc.shape}')

    if gyro.ndim != 2 or gyro.shape[1] != 3:
        raise ValueError(f'Formato inválido em {caminho_gyro.name}: {gyro.shape}')

    if len(acc) != len(gyro):
        raise ValueError(
            f'Comprimentos diferentes no experimento {numero_experimento}: '
            f'acc={len(acc)} e gyro={len(gyro)}.'
        )

    X_exp = np.column_stack((acc, gyro)).astype(np.float32)
    y_exp = np.zeros(len(X_exp), dtype=np.int8)
    cobertura = np.zeros(len(X_exp), dtype=np.int8)

    rotulos_exp = tabela_rotulos.loc[
        (tabela_rotulos['experimento'] == numero_experimento)
        & (tabela_rotulos['usuario'] == numero_usuario)
    ].sort_values('inicio')

    if rotulos_exp.empty:
        raise ValueError(
            f'Experimento {numero_experimento} do usuário '
            f'{numero_usuario} não possui rótulos.'
        )

    for linha in rotulos_exp.itertuples(index=False):
        inicio_python = int(linha.inicio) - 1
        fim_exclusivo = int(linha.fim)

        if inicio_python < 0 or fim_exclusivo > len(X_exp):
            raise IndexError(
                f'Intervalo [{linha.inicio}, {linha.fim}] fora dos '
                f'limites do experimento {numero_experimento}.'
            )

        y_exp[inicio_python:fim_exclusivo] = int(linha.atividade)
        cobertura[inicio_python:fim_exclusivo] += 1

    if cobertura.max() > 1:
        raise ValueError(
            f'Existem segmentos sobrepostos no experimento '
            f'{numero_experimento}.'
        )

    return X_exp, y_exp

## 8. Carregamento de todos os experimentos

Cada experimento será guardado como um dicionário dentro de uma lista. Essa organização conserva os limites naturais das sequências e evita que uma janela futura comece em um experimento e termine em outro.

In [26]:
dados_experimentos = []
registros_resumo = []

for numero_experimento, numero_usuario in pares_disponiveis:
    X_exp, y_exp = carregar_experimento_completo(
        caminho_acc=arquivos_acc[(numero_experimento, numero_usuario)],
        caminho_gyro=arquivos_gyro[(numero_experimento, numero_usuario)],
        numero_experimento=numero_experimento,
        numero_usuario=numero_usuario,
        tabela_rotulos=rotulos
    )

    dados_experimentos.append({
        'experimento': numero_experimento,
        'usuario': numero_usuario,
        'X': X_exp,
        'y': y_exp
    })

    registros_resumo.append({
        'experimento': numero_experimento,
        'usuario': numero_usuario,
        'amostras': len(X_exp),
        'duracao_segundos': len(X_exp) / FREQUENCIA_AMOSTRAGEM,
        'amostras_rotuladas': int(np.count_nonzero(y_exp)),
        'amostras_sem_rotulo': int(np.count_nonzero(y_exp == 0))
    })

resumo_experimentos = pd.DataFrame(registros_resumo)

print('Experimentos carregados:', len(dados_experimentos))
print('Usuários carregados:', resumo_experimentos['usuario'].nunique())
resumo_experimentos.head()

Experimentos carregados: 61
Usuários carregados: 30


,experimento,usuario,amostras,duracao_segundos,amostras_rotuladas,amostras_sem_rotulo
0,1,1,20598,411.96,13956,6642
1,2,1,19286,385.72,13949,5337
2,3,2,18026,360.52,12998,5028
3,4,2,16565,331.30,11666,4899
4,5,3,20994,419.88,13833,7161


## 9. Resumo por usuário

A tabela mostra quantos experimentos e quantas amostras estão associados a cada indivíduo.

In [28]:
resumo_usuarios = (
    resumo_experimentos
    .groupby('usuario', as_index=False)
    .agg(
        quantidade_experimentos=('experimento', 'nunique'),
        amostras=('amostras', 'sum'),
        duracao_segundos=('duracao_segundos', 'sum'),
        amostras_rotuladas=('amostras_rotuladas', 'sum'),
        amostras_sem_rotulo=('amostras_sem_rotulo', 'sum')
    )
)

resumo_usuarios['percentual_sem_rotulo'] = (
    100
    * resumo_usuarios['amostras_sem_rotulo']
    / resumo_usuarios['amostras']
)

resumo_usuarios.round(2)

,usuario,quantidade_experimentos,amostras,duracao_segundos,amostras_rotuladas,amostras_sem_rotulo,percentual_sem_rotulo
0,1,2,39884,797.68,27905,11979,30.03
1,2,2,34591,691.82,24664,9927,28.70
2,3,2,38487,769.74,27047,11440,29.72
3,4,2,33556,671.12,25392,8164,24.33
4,5,2,31902,638.04,24648,7254,22.74
5,6,2,48611,972.22,25681,22930,47.17
6,7,2,33223,664.46,24088,9135,27.50
7,8,2,31906,638.12,22876,9030,28.30
8,9,2,31865,637.30,23834,8031,25.20
9,10,3,37238,744.76,23419,13819,37.11


## 10. Construção das matrizes consolidadas

Também criaremos matrizes únicas para facilitar o salvamento. Os vetores `usuario`, `experimento` e `amostra_experimento` preservam a origem de cada linha.

A tabela `limites_experimentos` registra o início e o fim exclusivo de cada experimento na matriz consolidada. Esses limites serão essenciais para impedir que o janelamento atravesse duas sequências diferentes.

In [29]:
X = np.concatenate(
    [item['X'] for item in dados_experimentos],
    axis=0
).astype(np.float32)

y = np.concatenate(
    [item['y'] for item in dados_experimentos]
).astype(np.int8)

usuario = np.concatenate([
    np.full(len(item['y']), item['usuario'], dtype=np.int16)
    for item in dados_experimentos
])

experimento = np.concatenate([
    np.full(len(item['y']), item['experimento'], dtype=np.int16)
    for item in dados_experimentos
])

amostra_experimento = np.concatenate([
    np.arange(len(item['y']), dtype=np.int32)
    for item in dados_experimentos
])

comprimentos = np.array(
    [len(item['y']) for item in dados_experimentos],
    dtype=np.int32
)
fins_exclusivos = np.cumsum(comprimentos)
inicios = np.r_[0, fins_exclusivos[:-1]]

limites_experimentos = resumo_experimentos[[
    'experimento', 'usuario'
]].copy()
limites_experimentos['inicio_global'] = inicios
limites_experimentos['fim_global_exclusivo'] = fins_exclusivos

print('Formato de X:', X.shape)
print('Formato de y:', y.shape)
print('Formato de usuario:', usuario.shape)
print('Formato de experimento:', experimento.shape)
limites_experimentos.head()

Formato de X: (1122772, 6)
Formato de y: (1122772,)
Formato de usuario: (1122772,)
Formato de experimento: (1122772,)


,experimento,usuario,inicio_global,fim_global_exclusivo
0,1,1,0,20598
1,2,1,20598,39884
2,3,2,39884,57910
3,4,2,57910,74475
4,5,3,74475,95469


## 11. Verificações de consistência do dataset completo

As verificações abaixo confirmam dimensões, valores finitos, classes válidas e continuidade dos limites.

In [30]:
assert X.ndim == 2, 'X deve possuir duas dimensões nesta etapa.'
assert X.shape[1] == 6, 'X deve possuir seis características.'
assert len(X) == len(y) == len(usuario) == len(experimento)
assert len(X) == len(amostra_experimento)
assert np.isfinite(X).all(), 'X contém NaN ou infinito.'
assert set(np.unique(y)).issubset(set(range(13)))
assert resumo_experimentos['usuario'].nunique() == 30
assert len(resumo_experimentos) == 61
assert limites_experimentos.iloc[0]['inicio_global'] == 0
assert limites_experimentos.iloc[-1]['fim_global_exclusivo'] == len(X)
assert np.array_equal(
    limites_experimentos['inicio_global'].to_numpy()[1:],
    limites_experimentos['fim_global_exclusivo'].to_numpy()[:-1]
)

print('Todas as verificações foram concluídas com sucesso.')

Todas as verificações foram concluídas com sucesso.


## 12. Distribuição das classes no dataset completo

O percentual abaixo usa **todas as amostras** como denominador, inclusive a classe 0. Por isso, ele responde: “qual fração do dataset completo pertence a cada classe?”.

Também calcularemos um segundo percentual usando apenas as amostras rotuladas. Manter os dois denominadores explícitos evita comparar percentuais que representam universos diferentes.

In [31]:
contagens = np.bincount(y, minlength=13)
total_amostras = len(y)
total_rotuladas = int(np.count_nonzero(y))

percentual_total = 100 * contagens / total_amostras
percentual_rotulado = np.full(13, np.nan)
percentual_rotulado[1:] = 100 * contagens[1:] / total_rotuladas

distribuicao_classes = pd.DataFrame({
    'classe': np.arange(13),
    'atividade': [NOMES_ATIVIDADES[i] for i in range(13)],
    'quantidade_amostras': contagens,
    'duracao_segundos': contagens / FREQUENCIA_AMOSTRAGEM,
    'percentual_do_total': percentual_total,
    'percentual_das_rotuladas': percentual_rotulado
})

distribuicao_classes.round(2)

,classe,atividade,quantidade_amostras,duracao_segundos,percentual_do_total,percentual_das_rotuladas
0,0,sem rótulo,307158,6143.16,27.36,NaN
1,1,ficar em pé,122091,2441.82,10.87,14.97
2,2,sentar,116707,2334.14,10.39,14.31
3,3,deitar,107961,2159.22,9.62,13.24
4,4,caminhar,126677,2533.54,11.28,15.53
5,5,subir escadas,138105,2762.10,12.30,16.93
6,6,descer escadas,136865,2737.30,12.19,16.78
7,7,sentar → ficar em pé,10316,206.32,0.92,1.26
8,8,ficar em pé → sentar,8029,160.58,0.72,0.98
9,9,deitar → ficar em pé,12428,248.56,1.11,1.52


A coluna `percentual_das_rotuladas` não se aplica à classe 0, pois seu denominador contém apenas as classes de 1 a 12.

Neste notebook, manteremos `y = 0` para documentar fielmente a aquisição. A decisão de remover essas amostras será tomada antes do janelamento.

## 13. Inspeção de uma parte da matriz consolidada

Para não criar um `DataFrame` com todas as amostras apenas para visualização, construiremos uma tabela com as dez primeiras linhas.

In [32]:
N_LINHAS = 10

amostra_dataset = pd.DataFrame(
    X[:N_LINHAS],
    columns=NOMES_CARACTERISTICAS
)
amostra_dataset.insert(0, 'usuario', usuario[:N_LINHAS])
amostra_dataset.insert(1, 'experimento', experimento[:N_LINHAS])
amostra_dataset.insert(2, 'amostra_experimento', amostra_experimento[:N_LINHAS])
amostra_dataset['atividade'] = y[:N_LINHAS]

amostra_dataset

,usuario,experimento,amostra_experimento,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,atividade
0,1,1,0,0.918056,-0.112500,0.509722,-0.054978,-0.069639,-0.030849,0
1,1,1,1,0.911111,-0.093056,0.537500,-0.012523,0.019242,-0.038485,0
2,1,1,2,0.881944,-0.086111,0.513889,-0.023518,0.276417,0.006414,0
3,1,1,3,0.881944,-0.086111,0.513889,-0.093462,0.367741,0.001222,0
4,1,1,4,0.879167,-0.100000,0.505556,-0.124311,0.476780,-0.022907,0
5,1,1,5,0.888889,-0.105556,0.512500,-0.100487,0.519846,-0.067501,0
6,1,1,6,0.862500,-0.101389,0.509722,-0.149357,0.481056,-0.092546,0
7,1,1,7,0.861111,-0.104167,0.501389,-0.211054,0.389121,-0.074831,0
8,1,1,8,0.854167,-0.108333,0.527778,-0.222355,0.267864,-0.051924,0
9,1,1,9,0.851389,-0.101389,0.552778,-0.173791,0.207083,-0.032070,0


## 14. Salvamento dos dados processados

O arquivo `.npz` armazenará as matrizes NumPy de forma compactada. Os resumos serão salvos em `.csv`, pois são tabelas pequenas e podem ser abertos diretamente em outros programas.

A matriz consolidada **não autoriza** o janelamento direto sobre todas as linhas. No próximo notebook, as janelas deverão ser criadas separadamente para cada intervalo registrado em `limites_experimentos`.

In [33]:
CAMINHO_DATASET = PASTA_SAIDA / 'dataset_fase1_completo.npz'
CAMINHO_RESUMO_EXPERIMENTOS = (
    PASTA_SAIDA / 'resumo_experimentos.csv'
)
CAMINHO_RESUMO_USUARIOS = PASTA_SAIDA / 'resumo_usuarios.csv'
CAMINHO_DISTRIBUICAO = PASTA_SAIDA / 'distribuicao_classes.csv'
CAMINHO_LIMITES = PASTA_SAIDA / 'limites_experimentos.csv'

np.savez_compressed(
    CAMINHO_DATASET,
    X=X,
    y=y,
    usuario=usuario,
    experimento=experimento,
    amostra_experimento=amostra_experimento,
    nomes_caracteristicas=np.array(NOMES_CARACTERISTICAS)
)

resumo_experimentos.to_csv(
    CAMINHO_RESUMO_EXPERIMENTOS, index=False
)
resumo_usuarios.to_csv(CAMINHO_RESUMO_USUARIOS, index=False)
distribuicao_classes.to_csv(CAMINHO_DISTRIBUICAO, index=False)
limites_experimentos.to_csv(CAMINHO_LIMITES, index=False)

print('Arquivos salvos:')
for caminho in [
    CAMINHO_DATASET,
    CAMINHO_RESUMO_EXPERIMENTOS,
    CAMINHO_RESUMO_USUARIOS,
    CAMINHO_DISTRIBUICAO,
    CAMINHO_LIMITES
]:
    print('-', caminho)

Arquivos salvos:
- /content/drive/MyDrive/UCI-Motion-Database/DadosProcessados/dataset_fase1_completo.npz
- /content/drive/MyDrive/UCI-Motion-Database/DadosProcessados/resumo_experimentos.csv
- /content/drive/MyDrive/UCI-Motion-Database/DadosProcessados/resumo_usuarios.csv
- /content/drive/MyDrive/UCI-Motion-Database/DadosProcessados/distribuicao_classes.csv
- /content/drive/MyDrive/UCI-Motion-Database/DadosProcessados/limites_experimentos.csv


## 15. Teste de leitura do arquivo salvo

Por fim, abriremos o arquivo para verificar se as matrizes principais podem ser recuperadas corretamente.

In [34]:
with np.load(CAMINHO_DATASET) as dados_salvos:
    print('Variáveis armazenadas:', dados_salvos.files)
    print('X salvo:', dados_salvos['X'].shape, dados_salvos['X'].dtype)
    print('y salvo:', dados_salvos['y'].shape, dados_salvos['y'].dtype)

    assert dados_salvos['X'].shape == X.shape
    assert dados_salvos['y'].shape == y.shape

print('Teste de leitura concluído com sucesso.')

Variáveis armazenadas: ['X', 'y', 'usuario', 'experimento', 'amostra_experimento', 'nomes_caracteristicas']
X salvo: (1122772, 6) float32
y salvo: (1122772,) int8
Teste de leitura concluído com sucesso.


## 16. Conclusão

Neste notebook:

- identificamos automaticamente os 61 pares de arquivos;
- carregamos acelerômetro e giroscópio dos 30 usuários;
- criamos um rótulo para cada amostra;
- preservamos a identificação do usuário, do experimento e da amostra;
- verificamos formatos, limites, sobreposições e valores inválidos;
- analisamos a distribuição das classes com denominadores explícitos;
- salvamos o dataset completo e suas tabelas de apoio.

No próximo notebook, poderemos definir a estratégia de preparação: tratamento da classe 0, divisão por indivíduos, normalização baseada apenas no conjunto de treinamento e criação das janelas temporais para uma tarefa *many-to-one*.